# **Model 1 Classification Cocoa or other plans**

# STEP 1.1 — Setup

In [ ]:
#STEP 1.1 — Setup
#Mount Google Drive, install dependencies, and configure dataset paths.
# Define a helper function to run each pipeline stage.
import sys
import subprocess
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

# IMPORTANT: Update this path to match your Google Drive project folder.
# The project folder must contain the required scripts and dataset.
PROJECT = Path("/content/drive/MyDrive/cocoa-diseases")

SCRIPT = PROJECT / "script/model1_training/model1_training_script.py"
DATA = PROJECT / "dataset"
OUTPUT = PROJECT / "results/model1_initial_benchmark"
CACHE = Path("/content/model1_current_dataset")

if not SCRIPT.is_file():
    raise FileNotFoundError(f"Script not found: {SCRIPT}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pandas", "scikit-learn", "matplotlib", "Pillow", "psutil"],
    check=True,
)

def run_stage(stage, *extra, root=DATA):
    command = [
        sys.executable, "-B", "-u", str(SCRIPT),
        "--stage", stage,
        "--dataset-root", str(root),
        "--output-root", str(OUTPUT),
        *map(str, extra),
    ]

    print(f"Starting stage: {stage}", flush=True)

    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)

            return_code = process.wait()

        except KeyboardInterrupt:
            print("\nStopping the pipeline process...", flush=True)
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise

    if return_code != 0:
        raise RuntimeError(
            f"Stage '{stage}' failed with exit code {return_code}. "
            "See the output above."
        )

    print(f"Completed stage: {stage}", flush=True)

# STEP 1.2 - Preflight Validation : Model 1

In [ ]:
# 1.2.1 
# Validate the manifest, class mapping, image paths, and split integrity.
# Check pretrained models and preprocessing without training.
run_stage("check")

In [ ]:
#1.2.2 Summarize image counts by class ID, class name, and split.

import pandas as pd
from IPython.display import display

metadata = pd.read_csv(
    DATA / "model1_metadata.csv",
    dtype={"image_id": str, "class_id": int},
    keep_default_na=False,
)

summary = (
    metadata.groupby(["class_id", "class_name", "split"])
    .size()
    .unstack("split", fill_value=0)
    .reindex(columns=["train", "val", "test"], fill_value=0)
    .reset_index()
    .sort_values("class_id")
)

summary["total"] = summary[["train", "val", "test"]].sum(axis=1)
summary.columns.name = None

display(summary)
print(f"Total images: {summary['total'].sum():,}")

# STEP 1.3 - Runtime Cache : Model 1

In [ ]:
# Cache training and validation images on the Colab runtime disk.
# Reuse this cache across all four models to reduce Google Drive reads.
run_stage("cache", "--cache-root", CACHE)

# STEP 1.4 — Train and Compare : Model 1

In [ ]:
# Start a new benchmark or resume the latest compatible run automatically.
# Regenerate validation reports when all four models are complete.

import importlib.util
import json
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before training.")

if not (CACHE / "_cache_ready.json").is_file():
    raise RuntimeError("Prepare the runtime dataset cache before training.")

# Read the existing pipeline configuration without starting training.
spec = importlib.util.spec_from_file_location(
    "model1_benchmark_manager", SCRIPT
)
pipeline = importlib.util.module_from_spec(spec)
spec.loader.exec_module(pipeline)

config = pipeline.load_config(
    dataset_root=str(CACHE),
    output_root=str(OUTPUT),
)
config["ram_cache"] = True

frame, manifest_sha, _ = pipeline.read_manifest(config)
expected_protocol = pipeline.protocol_id(config, manifest_sha)
expected_models = list(pipeline.MODEL_NAMES)
expected_mapping = dict(pipeline.CLASS_MAP)

OUTPUT.mkdir(parents=True, exist_ok=True)


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def compatible_run(folder):
    """Check dataset identity, class mapping, and fixed training settings."""
    info_path = folder / "run_info.json"

    if not info_path.is_file():
        return False

    try:
        info = read_json(info_path)
    except (OSError, ValueError) as error:
        raise RuntimeError(
            f"Cannot inspect run metadata: {folder}"
        ) from error

    return (
        info.get("format_version") == pipeline.FORMAT_VERSION
        and info.get("manifest_sha256") == manifest_sha
        and info.get("class_mapping") == expected_mapping
        and info.get("protocol") == expected_protocol
        and info.get("selected_models") == expected_models
    )


def completed_model(folder, model_name):
    """Accept completion only when metrics and required artifacts exist."""
    model_dir = folder / model_name
    metrics_path = model_dir / "metrics.json"

    if not metrics_path.is_file():
        return False

    result = read_json(metrics_path)

    valid = (
        result.get("status") == "COMPLETED"
        and result.get("protocol") == expected_protocol
        and result.get("class_mapping") == expected_mapping
        and result.get("stage") == "validation"
        and result.get("test_evaluated") is False
    )

    required = [
        model_dir / "best.pth",
        model_dir / "validation_predictions.csv",
        model_dir / "history.csv",
    ]

    if not valid or not all(path.is_file() for path in required):
        raise RuntimeError(
            f"Inconsistent completion artifacts: {model_dir}"
        )

    return True


# Select the latest compatible run; leave incompatible experiments untouched.
candidates = [
    folder
    for folder in OUTPUT.glob("run_*")
    if folder.is_dir() and compatible_run(folder)
]

candidates.sort(
    key=lambda folder: (
        read_json(folder / "run_info.json").get("created_utc", ""),
        folder.name,
    ),
    reverse=True,
)

RUN = candidates[0] if candidates else None

if RUN is None:
    print("No compatible run found. Starting a new four-model benchmark.")
    print("Evaluation split: VALIDATION. Test evaluation is disabled.")

    run_stage("train", "--ram-cache", root=CACHE)

else:
    completed = [
        name for name in expected_models
        if completed_model(RUN, name)
    ]

    print(f"Selected run: {RUN}")
    print("Dataset identity, class mapping, and configuration match.")
    print(f"Completed models: {len(completed)}/{len(expected_models)}")

    if len(completed) == len(expected_models):
        print("All models are complete. Regenerating validation reports.")
        run_stage("report", "--resume", RUN)

    else:
        print("Resuming unfinished models from available checkpoints.")
        print("Completed models will be skipped.")
        print("Work after the latest saved checkpoint may be repeated.")

        run_stage(
            "train",
            "--ram-cache",
            "--resume", RUN,
            root=CACHE,
        )

# Display the comparison and candidate selection after successful completion.
if RUN is None:
    updated = [
        folder
        for folder in OUTPUT.glob("run_*")
        if folder.is_dir() and compatible_run(folder)
    ]
    if updated:
        RUN = max(
            updated,
            key=lambda folder: (
                read_json(folder / "run_info.json").get("created_utc", ""),
                folder.name,
            ),
        )

if RUN is not None:
    comparison = RUN / "comparison.csv"
    selection = RUN / "winner_candidate.json"

    if comparison.is_file():
        print("\nEVALUATION SPLIT: VALIDATION")
        display(pd.read_csv(comparison))

    if selection.is_file():
        print("\nWinner candidate selection:")
        print(json.dumps(read_json(selection), indent=2))

    print(f"\nResults directory: {RUN}")

# STEP 1.5 - Regenerate Reports

In [ ]:
# Regenerate validation tables and figures from saved results without retraining.
run_stage("report", "--resume", RUN)

# STEP 1.6 - Grad-CAM

In [ ]:
#1.6 Generate validation Grad-CAM comparisons and winner error explanations.
# Use existing checkpoints without training or evaluating the test set.

import sys
import subprocess
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/cocoa-diseases")
XAI_SCRIPT = PROJECT / "script/model1_training/model1_xai_gradcam.py"
RUN = PROJECT / (
    "results/model1_initial_benchmark/"
    "run_20261101T115128Z_6c0d08"
)

CACHE_ROOT = Path("/content/model1_current_dataset")
IMAGE_ROOT = (
    CACHE_ROOT
    if (CACHE_ROOT / "_cache_ready.json").is_file()
    else PROJECT / "dataset"
)

if not XAI_SCRIPT.is_file():
    raise FileNotFoundError(f"Wait for Drive synchronization: {XAI_SCRIPT}")

command = [
    sys.executable, "-B", "-u", str(XAI_SCRIPT),
    "--run", str(RUN),
    "--dataset-root", str(IMAGE_ROOT),
    "--samples-per-class", "1",
    "--seed", "42",
]

with subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise

if code:
    raise RuntimeError(f"Grad-CAM failed with exit code {code}")

# STEP 1.7 - Hyperparameter Tuning EfficientNet-B0

In [ ]:
#1.7.1 — Hyperparameter Tuning Setup 
# Configure EfficientNet-B0 tuning independently of the benchmark cells.
import sys
import subprocess
from pathlib import Path

TUNING_PROJECT = Path("/content/drive/MyDrive/cocoa-diseases")
TUNING_SCRIPT = (
    TUNING_PROJECT
    / "script/model1_training/model1_efficientnet_tuning.py"
)
TUNING_CACHE = Path("/content/model1_current_dataset")
TUNING_OUTPUT = TUNING_PROJECT / "results/model1_efficientnet_tuning"

if not TUNING_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Wait for Drive synchronization: {TUNING_SCRIPT}"
    )

if not (TUNING_CACHE / "_cache_ready.json").is_file():
    raise RuntimeError("Prepare the runtime dataset cache first.")


def run_tuning_stage(stage):
    # Call the tuning script explicitly; never reuse the benchmark helper.
    command = [
        sys.executable, "-B", "-u", str(TUNING_SCRIPT),
        "--project", str(TUNING_PROJECT),
        "--dataset-root", str(TUNING_CACHE),
        "--output-root", str(TUNING_OUTPUT),
        "--stage", stage,
    ]

    print("SCRIPT:", TUNING_SCRIPT, flush=True)
    print("OUTPUT:", TUNING_OUTPUT, flush=True)

    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            code = process.wait()
        except KeyboardInterrupt:
            print("\nStopping tuning...", flush=True)
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise

    if code:
        raise RuntimeError(
            f"Tuning stage '{stage}' failed: exit code {code}"
        )

In [ ]:
#1.7.2 Train or resume the two EfficientNet-B0 learning-rate trials on CUDA.
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before training.")

print("GPU:", torch.cuda.get_device_name(0), flush=True)
run_tuning_stage("train")

# STEP 1.8 — Grad-CAM of the Tuned EfficientNet-B0

In [ ]:
# Generate validation Grad-CAM for the deployed tuned EfficientNet-B0.
from google.colab import drive
drive.mount("/content/drive")

import hashlib
import json
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torchvision import models, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image, ImageOps

PROJECT = Path("/content/drive/MyDrive/cocoa-diseases")
TUNED_RUN = (
    PROJECT / "results/model1_efficientnet_tuning"
    / "tuning_20261101T140343Z_a6c9c3" # Replace with your tuning run ID
)
BUNDLE = TUNED_RUN / "application_candidate"
WEIGHTS = BUNDLE / "best.pth"

# IMPORTANT: Check the selected trial folder under:
# results/model1_efficientnet_tuning/<tuning_run>/
# Update "trial_1_lr_0.0003" to match the trial used
# to produce the selected EfficientNet-B0 checkpoint.
# The validation predictions must come from the same checkpoint.
PREDICTIONS = (
    TUNED_RUN / "trial_1_lr_0.0003/efficientnet_b0" # Replace with your selected trial folder
    / "validation_predictions.csv"
)
MANIFEST = PROJECT / "dataset/model1_metadata.csv"

def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for path in (
    WEIGHTS, PREDICTIONS, MANIFEST,
    BUNDLE / "inference_config.json",
    TUNED_RUN / "final_evaluation/deployment_metadata.json",
):
    if not path.is_file():
        raise FileNotFoundError(path)

config = read_json(BUNDLE / "inference_config.json")
deployment = read_json(TUNED_RUN / "final_evaluation/deployment_metadata.json")
weight_hash = sha256(WEIGHTS)

assert weight_hash == config["weights_sha256"]
assert weight_hash == deployment["weights_sha256"]
assert config["architecture"] == "efficientnet_b0"
assert config["class_mapping"] == {"other_plants": 0, "cocoa": 1}
assert config["threshold"] == 0.5

expected_preprocessing = {
    "image_size": 224,
    "resize": "stretch",
    "interpolation": "bilinear",
    "color": "RGB",
    "exif": "upright",
    "mean": [0.485, 0.456, 0.406],
    "std": [0.229, 0.224, 0.225],
}
for key, value in expected_preprocessing.items():
    if config.get(key) != value:
        raise RuntimeError(f"Preprocessing mismatch: {key}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.efficientnet_b0(weights=None)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)

checkpoint = torch.load(
    WEIGHTS, map_location="cpu", weights_only=True
)
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.to(device).eval()

transform = transforms.Compose([
    transforms.Resize(
        (224, 224), interpolation=InterpolationMode.BILINEAR
    ),
    transforms.ToTensor(),
    transforms.Normalize(config["mean"], config["std"]),
])

# Use validation records only. Select correct examples reproducibly.
pred = pd.read_csv(PREDICTIONS, dtype={"image_id": str})
manifest = pd.read_csv(MANIFEST, dtype={"image_id": str})
validation = manifest.loc[
    manifest["split"].eq("val"),
    ["image_id", "image_path", "class_id"],
]

data = pred.merge(
    validation,
    on="image_id",
    suffixes=("_prediction", "_manifest"),
    validate="one_to_one",
)
if len(data) != len(pred):
    raise RuntimeError("Some prediction records are not in validation.")
if not data["true_class_id"].eq(data["class_id"]).all():
    raise RuntimeError("Validation labels differ from the manifest.")

correct = data["true_class_id"].eq(data["predicted_class_id"])
parts = []
for class_id in (0, 1):
    candidates = data[correct & data["true_class_id"].eq(class_id)]
    parts.append(
        candidates.sample(n=min(3, len(candidates)), random_state=42)
    )
parts.append(data[~correct])
selected = pd.concat(parts).drop_duplicates("image_id").copy()

# Read the exported dataset images, not unprocessed original images.
selected["resolved_path"] = selected["image_path_manifest"].map(
    lambda value: str(PROJECT / "dataset" / value)
)
missing = [
    path for path in selected["resolved_path"]
    if not Path(path).is_file()
]
if missing:
    raise FileNotFoundError(
        "Exported dataset images are missing. First missing files:\n"
        + "\n".join(missing[:5])
    )

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT = TUNED_RUN / f"gradcam_tuned_validation_{stamp}_{uuid.uuid4().hex[:6]}"
OUTPUT.mkdir(parents=True, exist_ok=False)
selected.to_csv(OUTPUT / "selected_images.csv", index=False)

# Capture the final convolutional feature map without disabling gradients.
captured = {}

def capture_features(module, inputs, output):
    captured["features"] = output

hook = model.features[-1].register_forward_hook(capture_features)
class_names = ["other_plants", "cocoa"]
records = []

try:
    for row in selected.itertuples(index=False):
        image_path = Path(row.resolved_path)
        with Image.open(image_path) as source:
            original = ImageOps.exif_transpose(source).convert("RGB")

        tensor = transform(original).unsqueeze(0).to(device)
        model.zero_grad(set_to_none=True)

        # Do not wrap this section in no_grad() or inference_mode().
        logits = model(tensor)
        probabilities = logits.softmax(dim=1)[0].detach().cpu().numpy()
        predicted_id = int(probabilities[1] >= config["threshold"])
        true_id = int(row.true_class_id)

        if predicted_id != int(row.predicted_class_id):
            raise RuntimeError(
                f"Prediction differs from saved validation for {row.image_id}. "
                "Check package versions and preprocessing before interpretation."
            )

        # Correct examples: predicted class.
        # Errors: both predicted-class and true-class attribution.
        targets = [predicted_id]
        if true_id != predicted_id:
            targets.append(true_id)

        maps = {}
        features = captured["features"]
        for index, target_id in enumerate(targets):
            gradients = torch.autograd.grad(
                outputs=logits[0, target_id],
                inputs=features,
                retain_graph=index < len(targets) - 1,
            )[0]
            channel_weights = gradients.mean(dim=(2, 3), keepdim=True)
            cam = (channel_weights * features).sum(dim=1, keepdim=True)
            cam = F.relu(cam)
            cam = F.interpolate(
                cam,
                size=(original.height, original.width),
                mode="bilinear",
                align_corners=False,
            )[0, 0]

            cam = cam.detach().cpu().numpy()
            cam -= cam.min()
            peak = float(cam.max())
            if peak > 0:
                cam /= peak

            maps[target_id] = cam
            np.save(
                OUTPUT / f"{row.image_id}_target_{target_id}_cam.npy",
                cam,
            )

        fig, axes = plt.subplots(
            1, 1 + len(targets),
            figsize=(5 * (1 + len(targets)), 5),
            squeeze=False,
        )
        axes = axes[0]
        axes[0].imshow(original)
        axes[0].set_title(
            f"ID: {row.image_id}\nTrue: {class_names[true_id]}"
        )
        for ax, target_id in zip(axes[1:], targets):
            ax.imshow(original)
            ax.imshow(
                maps[target_id],
                cmap="jet", alpha=0.4, vmin=0, vmax=1,
            )
            ax.set_title(
                f"Attribution target: {class_names[target_id]}\n"
                f"P(target): {probabilities[target_id]:.4f}"
            )
        for ax in axes:
            ax.axis("off")

        outcome = "correct" if predicted_id == true_id else "incorrect"
        fig.suptitle(
            f"Tuned EfficientNet-B0 | VALIDATION | {outcome}\n"
            "Individually normalized attribution; colors are not confidence."
        )
        fig.tight_layout()
        figure_path = OUTPUT / f"gradcam_{row.image_id}_{outcome}.png"
        fig.savefig(figure_path, dpi=250, bbox_inches="tight")
        plt.show()
        plt.close(fig)

        for target_id in targets:
            records.append({
                "image_id": row.image_id,
                "image_path": str(image_path),
                "image_sha256": sha256(image_path),
                "split": "VALIDATION",
                "true_class": class_names[true_id],
                "predicted_class": class_names[predicted_id],
                "p_cocoa": float(probabilities[1]),
                "attribution_target": class_names[target_id],
                "outcome": outcome,
                "empty_cam": bool(maps[target_id].max() == 0),
                "figure": str(figure_path),
            })
finally:
    hook.remove()

pd.DataFrame(records).to_csv(OUTPUT / "gradcam_records.csv", index=False)

protocol = {
    "architecture": "efficientnet_b0",
    "checkpoint": str(WEIGHTS),
    "checkpoint_sha256": weight_hash,
    "selected_configuration": deployment.get("selected_configuration"),
    "evaluation_split": "VALIDATION",
    "threshold": config["threshold"],
    "preprocessing": expected_preprocessing,
    "target_layer": "model.features[-1]",
    "target_score": "Pre-softmax class logit",
    "method": "Grad-CAM: spatial mean gradients, weighted features, ReLU",
    "selection": "Up to 3 correct images per class, seed 42; all saved errors",
    "selected_images": len(selected),
    "normalization": "Independent min-max normalization for each map",
    "device": str(device),
    "torch": str(torch.__version__),
    "torchvision": __import__("torchvision").__version__,
    "test_used": False,
    "training_performed": False,
}
(OUTPUT / "protocol.json").write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
(OUTPUT / "COMPLETED.json").write_text(
    json.dumps({"status": "COMPLETED", "images": len(selected)}, indent=2),
    encoding="utf-8",
)

print("\nCompleted:", len(selected), "validation images")
print("Checkpoint SHA256:", weight_hash)
print("Results:", OUTPUT)

# STEP 1.9 - Test Set Evaluation

In [ ]:
# Configure final evaluation independently of training and tuning helpers.
import os
import sys
import signal
import subprocess
from pathlib import Path
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

FINAL_PROJECT = Path("/content/drive/MyDrive/cocoa-diseases")
FINAL_SCRIPT = (
    FINAL_PROJECT
    / "script/model1_training/model1_final_evaluation.py"
)

# Path to the EfficientNet-B0 tuning results.
# IMPORTANT: Replace the run folder name below with your own
# completed tuning run before performing final evaluation.
# The run must contain the selected model checkpoint and configuration.

FINAL_TUNING_RUN = FINAL_PROJECT / (
    "results/model1_efficientnet_tuning/"
    "tuning_20261101T140343Z_a6c9c3" # Replace with your tuning run ID
)

if not FINAL_SCRIPT.is_file():
    raise FileNotFoundError(
        f"Wait for Drive synchronization: {FINAL_SCRIPT}"
    )


def run_final_evaluation(stage):
    command = [
        sys.executable, "-B", "-u", str(FINAL_SCRIPT),
        "--project", str(FINAL_PROJECT),
        "--dataset-root", str(FINAL_PROJECT / "dataset"),
        "--tuning-run", str(FINAL_TUNING_RUN),
        "--stage", stage,
    ]

    print("SCRIPT:", FINAL_SCRIPT, flush=True)

    with subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
        start_new_session=True,
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            code = process.wait()
        except KeyboardInterrupt:
            print("\nStopping final evaluation...", flush=True)
            try:
                os.killpg(process.pid, signal.SIGTERM)
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
                process.wait()
            except ProcessLookupError:
                pass
            raise

    if code:
        raise RuntimeError(
            f"Final evaluation failed: exit code {code}"
        )

In [ ]:
#1.9.1 Evaluate the frozen candidate on the test split using CUDA.
# Reuse committed batches after interruption; skip inference when complete.
run_final_evaluation("test")

In [ ]:
# 1.9.2 Regenerate test reports without repeating inference.
run_final_evaluation("report")

# **Model 2 Cocoa Desiease Detection**

# 2.1 — Model 2 Benchmark Setup

In [ ]:

# 2.1.1 Set up the Model 2 benchmark and reuse the existing image cache.
import os
import sys
import subprocess
from pathlib import Path
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")

M2_PROJECT = Path("/content/drive/MyDrive/cocoa-diseases")
M2_SCRIPT = M2_PROJECT / "script/model2_training/model2_training_script.py"
M2_CACHE = Path("/content/model1_current_dataset")
M2_MANIFEST = M2_PROJECT / "dataset/model2_metadata_detection.csv"

for path in (M2_SCRIPT, M2_MANIFEST, M2_CACHE / "_cache_ready.json"):
    if not path.is_file():
        raise FileNotFoundError(f"Required file not found: {path}")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "ultralytics==8.4.198", "pandas", "matplotlib", "PyYAML"],
    check=True,
)

def run_model2_stage(stage):
    command = [
        sys.executable, "-B", "-u", str(M2_SCRIPT),
        "--project", str(M2_PROJECT),
        "--image-cache", str(M2_CACHE),
        "--stage", stage,
    ]
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        try:
            process.wait(timeout=15)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()

    if code:
        raise RuntimeError(f"Model 2 stage failed: {stage}")

print("Model 2 setup ready.")

In [ ]:
#2.1.2 Check the software versions and GPU availability before training.
import platform
import sys
import subprocess
import importlib.metadata as metadata

print("Operating system:", platform.platform())
print("Python:", sys.version.replace("\n", " "))

for package in ["torch", "torchvision", "ultralytics"]:
    try:
        print(f"{package}:", metadata.version(package))
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")

try:
    import torch

    print("CUDA available:", torch.cuda.is_available())
    print("PyTorch CUDA build:", torch.version.cuda)

    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("GPU memory (GB):",
              round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
except Exception as error:
    print("GPU check error:", error)

try:
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
        capture_output=True,
        text=True,
        check=False,
    )
    print("nvidia-smi:")
    print(result.stdout.strip() or result.stderr.strip())
except Exception as error:
    print("nvidia-smi unavailable:", error)

# Step 2.2 — Training and Automatic Resumption

In [ ]:
# Step 2.2 — Training and Automatic Resumption (Model 2)
# Run this cell to start training all four detectors (YOLOv8m, YOLO11n, YOLO12n, and YOLO26n).
# If training is interrupted, rerun the setup and then this cell to resume
# from the saved last.pt checkpoint using the same settings and paths.
run_model2_stage("train")

# Step 2.3 — Locate the Model 2 benchmark results.

In [ ]:
# Step 2.3 — Locate the Model 2 Benchmark Results
# Run this cell after Step 2.2 completes.
# Read the benchmark output path from active_run.json and verify that the
# comparison table and the 2×2 confusion matrix are available.
# The next cell uses RUN to display these results.
import json
from pathlib import Path

BASE = M2_PROJECT / "results/model2_four_model_benchmark"
active_run_file = BASE / "active_run.json"

if not active_run_file.is_file():
    raise FileNotFoundError(
        "Model 2 benchmark run not found. Run Step 2.2 first."
    )

run_info = json.loads(active_run_file.read_text(encoding="utf-8"))
RUN = Path(run_info["run"])

for filename in ("comparison.csv", "confusion_matrix_normalized_2x2.png"):
    if not (RUN / filename).is_file():
        raise FileNotFoundError(f"Missing benchmark output: {RUN / filename}")

print("Benchmark results:", RUN)

# Step 2.4 — Display the Model 2 Validation Results

In [ ]:
# Step 2.4 — Display the Model 2 Validation Results
# Run this cell after Step 2.3 to display the saved results for all four detectors.
# The training script saves the figures. This cell only reads and displays them.

import json

import pandas as pd
from IPython.display import Image, display

models = ("yolov8m", "yolo11n", "yolo12n", "yolo26n")

print("Evaluation split: Validation")
print(f"Results directory: {RUN}")

print("\nOverall model comparison")
display(pd.read_csv(RUN / "comparison.csv"))

print("\nNormalized confusion matrices — all four models")
display(Image(filename=str(RUN / "confusion_matrix_normalized_2x2.png")))

for name in models:
    folder = RUN / name
    print(f"\n{'=' * 50}\nModel: {name}")

    if (folder / "COMPLETED.json").is_file():
        print("Status: Validation reporting completed.")
    elif (folder / "TRAINING_FINISHED.json").is_file():
        print("Status: Training finished; validation reporting is incomplete.")
    else:
        print("Status: Completion is not confirmed.")

    metrics_file = folder / "validation_metrics.json"
    if metrics_file.is_file():
        print("\nSaved validation metrics")
        metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
        print(json.dumps(metrics, indent=2, ensure_ascii=False))

    per_class_file = folder / "per_class_validation.csv"
    if per_class_file.is_file():
        print("\nPer-class validation results")
        display(pd.read_csv(per_class_file))

    history_file = folder / "results.csv"
    if history_file.is_file():
        print("\nLast five recorded training epochs")
        history = pd.read_csv(history_file)
        history.columns = history.columns.str.strip()
        display(history.tail(5))

    figures = folder / "publication_figures"
    for filename, title in (
        ("confusion_matrix.png", "Confusion matrix (counts)"),
        ("confusion_matrix_normalized.png", "Normalized confusion matrix"),
    ):
        figure_file = figures / filename
        if figure_file.is_file():
            print(f"\n{title} — {name}")
            display(Image(filename=str(figure_file)))
        else:
            print(f"Missing figure: {figure_file}")

In [ ]:
# Step 2.4.1 — Validation Error Analysis
# Match saved YOLO26n validation predictions to ground-truth boxes.

import subprocess
import sys

review_script = M2_PROJECT / "script/model2_training/review_saved_predictions.py"

subprocess.run(
    [
        sys.executable, "-u", str(review_script),
        "--project", str(M2_PROJECT),
        "--benchmark-run", str(RUN),
    ],
    check=True,
)

errors_csv = RUN / "yolo26n" / "validation_error_review" / "wrong_class_boxes.csv"
print("Wrong-class detections:", errors_csv)

In [ ]:
# Step 2.4.2 — Grad-CAM Analysis
# Generate Grad-CAM overlays for selected YOLO26n validation errors.

import subprocess
import sys

gradcam_script = M2_PROJECT / "script/model2_training/model2_yolo26_gradcam.py"

subprocess.run(
    [
        sys.executable, "-u", str(gradcam_script),
        "--project", str(M2_PROJECT),
        "--benchmark-run", str(RUN),
        "--errors", str(errors_csv),
    ],
    check=True,
)

# Step 2.5 — Evaluate the Selected YOLO26n Checkpoint on the Held-Out Test Set

In [ ]:
# Step 2.5 — Evaluate the Selected YOLO26n Checkpoint on the Held-Out Test Set
# Run this cell after validation-based model selection is complete.
# Use the YOLO26n checkpoint from the benchmark run identified in Step 2.3.

test_script = M2_PROJECT / "script/model2_training/model2_yolo26_final_test.py"
checkpoint = RUN / "yolo26n" / "weights" / "best.pt"

for path in (test_script, checkpoint):
    if not path.is_file():
        raise FileNotFoundError(f"Required file not found: {path}")

subprocess.run(
    [
        sys.executable, "-u", str(test_script),
        "--project", str(M2_PROJECT),
        "--benchmark-run", str(RUN),
    ],
    check=True,
)

# Step 2.6 — Group-Aware Five-Fold Results

In [ ]:
# Step 2.6 — Group-Aware Five-Fold Results
# Verify the fold assignments, train or resume all five folds, and display the saved results.
import subprocess
import sys
import json
import pandas as pd
from IPython.display import display
cv_script = M2_PROJECT / "script/model2_training/model2_yolo26_grouped_5fold.py"
baseline_run = M2_PROJECT / "results/model2_yolo26_benchmark/run_20261102T111137Z_c24e0f"
command = [
    sys.executable, "-u", str(cv_script),
    "--project", str(M2_PROJECT),
    "--data", "/content/model2_yolo26_dataset/data.yaml",
    "--baseline-run", str(baseline_run),
]
subprocess.run([*command, "--plan-only"], check=True)
subprocess.run(command, check=True)

# Continue with the results display code, starting at RESULT_DIR.
RESULT_DIR = M2_PROJECT / "results/model2_yolo26_grouped_5fold"
ASSIGNMENT_FILE = RESULT_DIR / "fold_assignments.csv"
# Load and validate the fold assignments.
if not ASSIGNMENT_FILE.is_file():
    raise FileNotFoundError(f"Fold assignment file not found: {ASSIGNMENT_FILE}")
assignments = pd.read_csv(
    ASSIGNMENT_FILE,
    dtype={"image_id": str, "group": str},
    keep_default_na=False,
)
assignments["validation_fold"] = pd.to_numeric(
    assignments["validation_fold"], errors="raise"
)
if assignments["image_id"].duplicated().any():
    raise ValueError("The assignment file contains duplicate image IDs.")
if not assignments["validation_fold"].isin(range(1, 6)).all():
    raise ValueError("Validation fold numbers must be between 1 and 5.")
if assignments["group"].str.strip().eq("").any():
    raise ValueError("The assignment file contains missing group IDs.")
groups_across_folds = (
    assignments.groupby("group")["validation_fold"].nunique().gt(1).sum()
)
if groups_across_folds:
    raise ValueError(
        f"{groups_across_folds} image groups are assigned to multiple validation folds."
    )
total_images = len(assignments)
records = []
# Read the saved metrics and count the unique images in each fold.
for fold in range(1, 6):
    metrics_file = RESULT_DIR / f"fold_{fold}" / "metrics.json"
    if not metrics_file.is_file():
        raise FileNotFoundError(f"Metrics file for fold {fold} not found: {metrics_file}")
    metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
    if int(metrics["fold"]) != fold:
        raise ValueError(f"The fold number in {metrics_file} does not match fold {fold}.")
    validation_images = int(assignments["validation_fold"].eq(fold).sum())
    if validation_images == 0:
        raise ValueError(f"Fold {fold} contains no validation images.")
    records.append({
        "Fold": fold,
        "Training images": total_images - validation_images,
        "Validation images": validation_images,
        "Precision (%)": float(metrics["precision"]) * 100,
        "Recall (%)": float(metrics["recall"]) * 100,
        "mAP@0.50 (%)": float(metrics["map50"]) * 100,
        "mAP@0.50:0.95 (%)": float(metrics["map50_95"]) * 100,
    })
results = pd.DataFrame(records).set_index("Fold")
metric_columns = [
    "Precision (%)",
    "Recall (%)",
    "mAP@0.50 (%)",
    "mAP@0.50:0.95 (%)",
]
# Display image counts and performance metrics for each fold.
print("YOLO26n Group-Aware Five-Fold Cross-Validation")
print(f"Total unique images: {total_images:,}")
print("Each image group was assigned to a single validation fold.")
display(results.style.format({
    "Training images": "{:,.0f}",
    "Validation images": "{:,.0f}",
    **{column: "{:.2f}" for column in metric_columns},
}))
# Calculate the mean and sample standard deviation across the five folds.
summary = pd.DataFrame({
    "Mean (%)": results[metric_columns].mean(),
    "SD (percentage points)": results[metric_columns].std(ddof=1),
})
summary.index.name = "Metric"
summary["Mean ± SD"] = [
    f"{mean:.2f} ± {sd:.2f}"
    for mean, sd in zip(summary["Mean (%)"], summary["SD (percentage points)"])
]
print("Mean and standard deviation across the five folds")
display(summary.style.format({
    "Mean (%)": "{:.2f}",
    "SD (percentage points)": "{:.2f}",
}))
# Show the first ten assignments and explain how to reproduce each split.
print("Example image assignments")
display(assignments.head(10))
print(
    "To reproduce the split for fold k:\n"
    "• Use images with validation_fold equal to k for validation.\n"
    "• Use images with validation_fold different from k for training.\n"
    "• Match each image_id to the dataset metadata to locate its image and label files."
)
print(f"Full fold assignment file: {ASSIGNMENT_FILE}")

# Step 2.7 — Display the YOLO26n Held-Out Test Results

In [ ]:
# Step 2.6 — Display the YOLO26n Held-Out Test Results
# Run this cell after Step 2.5 completes.
# Read the saved test metrics for the benchmark run identified in Step 2.3.

import json

import pandas as pd
from IPython.display import display

out = M2_PROJECT / "results/model2_yolo26_final_test" / RUN.name

for filename in ("COMPLETED.json", "test_metrics.json", "per_class_test.csv"):
    if not (out / filename).is_file():
        raise FileNotFoundError(
            f"Missing test output: {out / filename}. Complete Step 2.5 first."
        )

metrics = json.loads((out / "test_metrics.json").read_text(encoding="utf-8"))

print("Evaluation split:", metrics["evaluation_split"])
print("Images:", metrics["images"])
print("Ground-truth boxes:", metrics["ground_truth_boxes"])

print("\nOverall test performance")
columns = ["mAP50_95", "mAP50", "precision", "recall"]
display(pd.DataFrame([metrics])[columns].round(4))

print("\nPer-class test performance")
display(pd.read_csv(out / "per_class_test.csv").round(4))

# Step 2.7 # grouped_5fold

In [ ]:
# Verify the grouped splits, then train or resume all five folds.
import subprocess
import sys

command = [
    sys.executable, "-u",
    "/content/drive/MyDrive/cocoa_diseases_final/model2_yolo26_grouped_5fold.py",
    "--project", "/content/drive/MyDrive/cocoa-diseases",
    "--data", "/content/model2_yolo26_dataset/data.yaml",
]

subprocess.run(command + ["--plan-only"], check=True)
subprocess.run(command, check=True)